In [1]:
from flask import Flask, request, jsonify, render_template_string
import tensorflow as tf
import numpy as np
import os
import requests
import numpy as np
from flask import Flask, request, render_template_string, jsonify
import numpy as np
import pickle

In [2]:
app = Flask(__name__)
with open(r"C:\onedrive\Desktop\Senior Proejct\Project\tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

In [3]:

# Load the model
MODEL_PATH = r"phishing_model.keras"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)




In [ ]:
# Simple HTML page
HOME_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Neural Network API</title>
    <style>
        body { font-family: Arial; background: #f4f6f8; text-align: center; padding-top: 50px; }
        .box { background: white; padding: 40px; margin: auto; width: 500px; border-radius: 12px; box-shadow: 0 5px 15px rgba(0,0,0,0.1); }
        h1 { color: #333; }
        textarea { width: 100%; height: 150px; margin-top: 10px; font-size: 1em; }
        button { padding: 10px 20px; margin-top: 10px; font-size: 1em; border-radius: 5px; background: #4CAF50; color: white; border: none; cursor: pointer; }
        button:hover { background: #45a049; }
        .result { margin-top: 20px; font-weight: bold; }
    </style>
</head>
<body>
    <div class="box">
        <h1> Neural Network API</h1>
        <p>Paste your email below and get a prediction:</p>
        <form action="/predict" method="post">
            <textarea name="email_text" placeholder="Enter email here..."></textarea>
            <br>
            <button type="submit">Predict</button>
        </form>
        {% if prediction_label %}
        <div class="result">
            Prediction: {{ prediction_label }} (Probability: {{ prediction_prob | round(3) }})
        </div>
        {% endif %}
    </div>
</body>
</html>
"""


@app.route('/', methods=['GET', 'POST'])
def home():
    prediction_label = None
    prediction_prob = None
    if request.method == 'POST':
        # get email text from form
        email_text = request.form.get('email_text', '')

        # convert email text to model input
        email_text = request.form.get('email_text', '')
        features = vectorizer.transform([email_text]).toarray().astype(np.float32)
        prediction_prob = float(model.predict(features)[0][0])

        # Convert probability to label
        prediction_label = "Phishing" if prediction_prob >= 0.5 else "Safe"

    return render_template_string(HOME_HTML, prediction_label=prediction_label, prediction_prob=prediction_prob)


@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Check if the request is JSON or form
        if request.is_json:
            data = request.get_json()
            email_text = data['email_text']
        else:
            email_text = request.form.get('email_text', '')
        
        features = vectorizer.transform([email_text]).toarray().astype(np.float32)
        preds = model.predict(features)
        pred_prob = float(model.predict(features)[0][0])
        pred_label = "Phishing" if pred_prob >= 0.5 else "Safe"

        return jsonify({
            "prediction_label": pred_label,
            })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Nov/2025 16:42:10] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


127.0.0.1 - - [05/Nov/2025 16:42:29] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


...

127.0.0.1 - - [05/Nov/2025 16:43:08] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 16:44:48] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


127.0.0.1 - - [05/Nov/2025 16:45:04] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step


127.0.0.1 - - [05/Nov/2025 16:45:17] "POST /predict HTTP/1.1" 200 -
